# GSE173958: Lineage-Validated MOT-MIC Inference



## Dataset

This tutorial follows the scTour-style tutorial layout: dataset, model training, inference, visualization, and robustness.

`GSE173958` is the strongest validation dataset for MOT-MIC because it contains paired single-cell transcriptomes and CRISPR lineage tracing in metastatic PDAC. Primary tumor cells can be mapped to liver, lung, macrometastasis, and CTC targets. Aggressive clone labels provide the closest available gold standard for MIC prediction.


In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
repo_root = cwd.parent if cwd.name == 'notebooks' else cwd
sys.path.insert(0, str(repo_root))

import pandas as pd
import matplotlib.pyplot as plt

from motmic import MOTMIC, evaluate_binary_labels, find_10x_triplet, read_10x_mtx, downsample_cells
from motmic.interpret import rank_genes_with_shap



## Data Download

Inspect the GEO supplementary file list first. The full dataset is large, so the command below is shown as a controlled download entry point. Run without `--dry-run` when you are ready to download the real matrices.


In [ ]:
# From the repository root:
# !python scripts/download_geo.py --dry-run --from-filelist --gse GSE173958
# !python scripts/download_geo.py --from-filelist --gse GSE173958

raw_dir = repo_root / 'data' / 'raw' / 'GSE173958'
raw_dir


## Data Loading

The example below uses mouse M1 primary tumor as the source and liver/lung/macrometastasis samples as metastatic targets. Downsampling keeps the tutorial fast; remove it for full analysis.


In [ ]:
# Uncomment after downloading GSE173958 files.
# pt_triplet = find_10x_triplet(raw_dir, 'M1-PT')
# liver_triplet = find_10x_triplet(raw_dir, 'M1-Liver')
# lung_triplet = find_10x_triplet(raw_dir, 'M1-Lung')
# met_triplet = find_10x_triplet(raw_dir, 'M1-Met')
#
# primary = downsample_cells(read_10x_mtx(*pt_triplet), n_cells=2500)
# metastases = {
#     'liver': downsample_cells(read_10x_mtx(*liver_triplet), n_cells=1500),
#     'lung': downsample_cells(read_10x_mtx(*lung_triplet), n_cells=1500),
#     'macromet': downsample_cells(read_10x_mtx(*met_triplet), n_cells=1500),
# }
# primary.shape, {k: v.shape for k, v in metastases.items()}


## Model Training

MOT-MIC learns a shared PCA latent space internally and then performs organ-specific unbalanced optimal transport. The parameters `epsilon`, `rho`, and `top_k` control transport smoothness, marginal relaxation, and origin sparsity.


In [ ]:
# model = MOTMIC(n_components=30, epsilon=0.08, rho=1.2, top_k=1)
# result = model.fit_predict(primary, metastases)
# scores = result.to_frame()
# scores.head()


## MIC Inference

The key output is a primary-cell score table with organ-specific and pan-metastatic MIC scores.


In [ ]:
# scores[['liver', 'lung', 'macromet', 'pan_MIC_score', 'organ_specificity', 'predicted_site']].head()


## Visualization

Use rank plots and organ-score distributions to inspect whether top primary cells have a focused or broad metastatic destination profile.


In [ ]:
# fig, ax = plt.subplots(figsize=(6, 4))
# scores['pan_MIC_score'].sort_values(ascending=False).reset_index(drop=True).plot(ax=ax)
# ax.set_xlabel('Primary cells ranked by MOT-MIC score')
# ax.set_ylabel('pan-MIC score')
# ax.set_title('GSE173958 primary-cell MIC ranking')
# plt.show()


## Lineage Validation

Load clone/aggressive-clone annotations when available from the original lineage analysis. The expected format is a table indexed by cell barcode with a boolean column such as `aggressive_clone`.


In [ ]:
# clone_labels = pd.read_csv(repo_root / 'data/processed/GSE173958_clone_labels.csv', index_col=0)
# metrics = evaluate_binary_labels(result.pan_score, clone_labels['aggressive_clone'])
# metrics


## SHAP Gene Ranking

After MOT-MIC inference, train an interpretable model to distinguish high-MIC and low-MIC primary cells. SHAP values prioritize candidate genes; combine them with lineage enrichment for the final ranked list.


In [ ]:
# high_mic = result.pan_score >= result.pan_score.quantile(0.9)
# gene_ranking = rank_genes_with_shap(primary, high_mic.astype(int))
# gene_ranking.head(30)


## Robustness

Repeat inference across random seeds, `epsilon`, `rho`, and `top_k`. Stable MICs should recur in the top-ranked primary cells and remain enriched in aggressive clones.


In [ ]:
# robustness = []
# for eps in [0.04, 0.08, 0.12]:
#     model = MOTMIC(n_components=30, epsilon=eps, rho=1.2, top_k=1)
#     res = model.fit_predict(primary, metastases)
#     robustness.append(res.pan_score.rename(f'epsilon_{eps}'))
# pd.concat(robustness, axis=1).corr()
